In [1]:
print("Google Colab Environment Initialized Successfully!")



Google Colab Environment Initialized Successfully!


In [2]:
# =====================================================================
# STEP 1: KNOWLEDGE BASE (KB)
# Declarative facts and decision rules representing university grading policy.
# =====================================================================

UNIVERSITY_KB = {
    "system_name": "BSCS Academic Performance & Grading Expert System",
    "passing_marks": 50,
    "rules": [
        {
            "id": "RULE_01",
            "conditions": {"attendance_percentage": 74},  # Below 75%
            "condition_type": "attendance_below",
            "grade": "F",
            "status": "Debarred",
            "action": "Ineligible to sit for final exam due to short attendance."
        },
        {
            "id": "RULE_02",
            "conditions": {"marks_min": 85, "marks_max": 100},
            "condition_type": "marks_range",
            "grade": "A",
            "status": "Passed with Distinction",
            "action": "Recommended for Merit Scholarship."
        },
        {
            "id": "RULE_03",
            "conditions": {"marks_min": 75, "marks_max": 84},
            "condition_type": "marks_range",
            "grade": "B",
            "status": "Passed",
            "action": "Good standing. Maintain momentum."
        },
        {
            "id": "RULE_04",
            "conditions": {"marks_min": 60, "marks_max": 74},
            "condition_type": "marks_range",
            "grade": "C",
            "status": "Passed",
            "action": "Satisfactory performance. Focus on weak subject areas."
        },
        {
            "id": "RULE_05",
            "conditions": {"marks_min": 50, "marks_max": 59},
            "condition_type": "marks_range",
            "grade": "D",
            "status": "Conditional Pass",
            "action": "Warning: Near failure threshold. Academic counseling advised."
        },
        {
            "id": "RULE_06",
            "conditions": {"marks_min": 0, "marks_max": 49},
            "condition_type": "marks_range",
            "grade": "F",
            "status": "Failed",
            "action": "Must repeat the course in the upcoming semester."
        }
    ]
}

print(f"Loaded: {UNIVERSITY_KB['system_name']}")

Loaded: BSCS Academic Performance & Grading Expert System


In [3]:
# =====================================================================
# STEP 2: INFERENCE ENGINE
# Evaluates student records against rules stored in the Knowledge Base.
# =====================================================================

def evaluate_student(student_record, kb):
    """
    Evaluates student marks and attendance against university rules.
    Returns computed grade, academic status, and recommended action.
    """
    print(f"\n--- EVALUATING STUDENT: {student_record['student_name']} (ID: {student_record['student_id']}) ---")

    # Check rule 1: Attendance constraint first
    if student_record["attendance"] < 75:
        attendance_rule = next(r for r in kb["rules"] if r["condition_type"] == "attendance_below")
        return {
            "rule_id": attendance_rule["id"],
            "grade": attendance_rule["grade"],
            "status": attendance_rule["status"],
            "action": attendance_rule["action"]
        }

    # Check grade spectrum rules
    marks = student_record["total_marks"]
    for rule in kb["rules"]:
        if rule["condition_type"] == "marks_range":
            if rule["conditions"]["marks_min"] <= marks <= rule["conditions"]["marks_max"]:
                return {
                    "rule_id": rule["id"],
                    "grade": rule["grade"],
                    "status": rule["status"],
                    "action": rule["action"]
                }

    return {
        "rule_id": "UNKNOWN",
        "grade": "N/A",
        "status": "Invalid Score Entry",
        "action": "Verify submitted marks with the exam department."
    }

In [4]:
# =====================================================================
# STEP 3: WORKING MEMORY & TEST EXECUTION
# =====================================================================

# Test Case 1: High scorer with full attendance
student_1 = {
    "student_id": "BSCS-2024-012",
    "student_name": "Ali Khan",
    "total_marks": 88,
    "attendance": 92
}

# Test Case 2: Good marks, but failed attendance threshold
student_2 = {
    "student_id": "BSCS-2024-045",
    "student_name": "Sara Ahmed",
    "total_marks": 81,
    "attendance": 68
}

# Run Inference Engine
result_1 = evaluate_student(student_1, UNIVERSITY_KB)
print(f"Grade: {result_1['grade']} | Status: {result_1['status']}")
print(f"Advisory Action: {result_1['action']}")

result_2 = evaluate_student(student_2, UNIVERSITY_KB)
print(f"Grade: {result_2['grade']} | Status: {result_2['status']}")
print(f"Advisory Action: {result_2['action']}")


--- EVALUATING STUDENT: Ali Khan (ID: BSCS-2024-012) ---
Grade: A | Status: Passed with Distinction
Advisory Action: Recommended for Merit Scholarship.

--- EVALUATING STUDENT: Sara Ahmed (ID: BSCS-2024-045) ---
Grade: F | Status: Debarred
Advisory Action: Ineligible to sit for final exam due to short attendance.


In [5]:
# ============================================================
# EXERCISE 1: DEGREE HONORS RULE
# ============================================================

degree_honors_rule = {
    "id": "RULE_07",
    "conditions": {
        "cgpa_min": 3.70,
        "disciplinary_warnings": 0
    },
    "condition_type": "degree_honors",
    "status": "Graduated with Magna Cum Laude Honors"
}

# Add rule only once
if not any(r["id"] == "RULE_07" for r in UNIVERSITY_KB["rules"]):
    UNIVERSITY_KB["rules"].append(degree_honors_rule)

print("Degree Honors Rule Added Successfully!")


# ============================================================
# EXERCISE 2: TRACING FORWARD CHAINING
# ============================================================

def evaluate_student(student_record, kb):

    print("\n--- FORWARD CHAINING ---")
    print("Evaluating:", student_record["student_name"])

    marks_result = None

    for rule in kb["rules"]:

        print("\nChecking:", rule["id"])

        # Rule 1: Attendance
        if rule["condition_type"] == "attendance_below":

            condition = student_record["attendance"] < 75
            print("Attendance below 75%:", condition)

            if condition:
                return {
                    "rule_id": rule["id"],
                    "grade": rule["grade"],
                    "status": rule["status"],
                    "action": rule["action"]
                }

        # Rule 2-6: Marks
        elif rule["condition_type"] == "marks_range":

            marks = student_record["total_marks"]

            condition = (
                rule["conditions"]["marks_min"]
                <= marks
                <= rule["conditions"]["marks_max"]
            )

            print("Marks condition:", condition)

            if condition:
                marks_result = {
                    "rule_id": rule["id"],
                    "grade": rule["grade"],
                    "status": rule["status"],
                    "action": rule["action"]
                }

        # Rule 7: Degree Honors
        elif rule["condition_type"] == "degree_honors":

            cgpa = student_record.get("cgpa", 0)
            warnings = student_record.get(
                "disciplinary_warnings", -1
            )

            condition = (
                cgpa >= rule["conditions"]["cgpa_min"]
                and warnings == rule["conditions"]["disciplinary_warnings"]
            )

            print("CGPA condition:", cgpa >= 3.70)
            print("Disciplinary condition:", warnings == 0)

            if condition:
                return {
                    "rule_id": rule["id"],
                    "grade": "N/A",
                    "status": rule["status"],
                    "action": "Degree honors awarded."
                }

    if marks_result:
        return marks_result

    return {
        "rule_id": "UNKNOWN",
        "grade": "N/A",
        "status": "Invalid Score Entry",
        "action": "Verify student record."
    }


# ============================================================
# TESTING THE NEW RULE
# ============================================================

student_3 = {
    "student_id": "BSCS-2024-100",
    "student_name": "Test Student",
    "total_marks": 88,
    "attendance": 90,
    "cgpa": 3.85,
    "disciplinary_warnings": 0
}

result_3 = evaluate_student(student_3, UNIVERSITY_KB)

print("\n========== FINAL RESULT ==========")
print("Rule:", result_3["rule_id"])
print("Status:", result_3["status"])
print("Action:", result_3["action"])

Degree Honors Rule Added Successfully!

--- FORWARD CHAINING ---
Evaluating: Test Student

Checking: RULE_01
Attendance below 75%: False

Checking: RULE_02
Marks condition: True

Checking: RULE_03
Marks condition: False

Checking: RULE_04
Marks condition: False

Checking: RULE_05
Marks condition: False

Checking: RULE_06
Marks condition: False

Checking: RULE_07
CGPA condition: True
Disciplinary condition: True

========== FINAL RESULT ==========
Rule: RULE_07
Status: Graduated with Magna Cum Laude Honors
Action: Degree honors awarded.


### Additional test cases for the Degree Honors rule
The honors rule must fire only when **both** conditions hold (`cgpa >= 3.70` and `disciplinary_warnings == 0`). These cases confirm the rule does *not* fire when either condition fails, and that the attendance rule still takes priority.

In [6]:
tests = [
    ("CGPA below 3.70",           {"student_id": "T-1", "student_name": "Low CGPA",    "total_marks": 88, "attendance": 90, "cgpa": 3.50, "disciplinary_warnings": 0}),
    ("Has a disciplinary warning", {"student_id": "T-2", "student_name": "Warned",      "total_marks": 88, "attendance": 90, "cgpa": 3.85, "disciplinary_warnings": 1}),
    ("Short attendance, high CGPA", {"student_id": "T-3", "student_name": "Absent",     "total_marks": 88, "attendance": 60, "cgpa": 3.85, "disciplinary_warnings": 0}),
]

for label, record in tests:
    r = evaluate_student(record, UNIVERSITY_KB)
    print(f"\n>>> {label}: rule = {r['rule_id']} | grade = {r['grade']} | status = {r['status']}")


--- FORWARD CHAINING ---
Evaluating: Low CGPA

Checking: RULE_01
Attendance below 75%: False

Checking: RULE_02
Marks condition: True

Checking: RULE_03
Marks condition: False

Checking: RULE_04
Marks condition: False

Checking: RULE_05
Marks condition: False

Checking: RULE_06
Marks condition: False

Checking: RULE_07
CGPA condition: False
Disciplinary condition: True

>>> CGPA below 3.70: rule = RULE_02 | grade = A | status = Passed with Distinction

--- FORWARD CHAINING ---
Evaluating: Warned

Checking: RULE_01
Attendance below 75%: False

Checking: RULE_02
Marks condition: True

Checking: RULE_03
Marks condition: False

Checking: RULE_04
Marks condition: False

Checking: RULE_05
Marks condition: False

Checking: RULE_06
Marks condition: False

Checking: RULE_07
CGPA condition: True
Disciplinary condition: False

>>> Has a disciplinary warning: rule = RULE_02 | grade = A | status = Passed with Distinction

--- FORWARD CHAINING ---
Evaluating: Absent

Checking: RULE_01
Attendance be

**Result:** CGPA below 3.70 and a disciplinary warning both fall through to the marks-based rule (RULE_02, grade A), so honors is not awarded. A student with short attendance is debarred by RULE_01 before any other rule is considered, even with a high CGPA.

**Conclusion:** The system separates the *Knowledge Base* (rules as data in `UNIVERSITY_KB`) from the *Inference Engine* (`evaluate_student`), so RULE_07 was added to the KB as one new dictionary. Because it introduces a new condition type (`degree_honors`), the engine also needed one new branch, which is the typical cost of adding a new kind of rule. The forward-chaining trace prints each rule condition in order, showing step-by-step deduction.